In [4]:
#Bibliotheken importieren:

import requests
from bs4 import BeautifulSoup as soup
import unicodedata

In [1]:
#SRU-Abfrage:

base_url = "https://services.dnb.de/sru/dnb"

In [2]:
def enquiry(searchterm, dataform): 
    
    parameter = {'version' : '1.1' , 
                 'operation' : 'searchRetrieve' , 
                 'query' : searchterm, 
                 'recordSchema' : dataform,
                 'maximumRecords': '100'} 
    
    r1 = requests.get(base_url, params = parameter)  
    
    return r1  

In [6]:
testresult = enquiry("Bibliothekswesen and location=onlinefree", "MARC21-xml")

In [7]:
print(testresult.content)

b'<?xml version="1.0" encoding="UTF-8"?>\n<searchRetrieveResponse xmlns="http://www.loc.gov/zing/srw/"><version>1.1</version><numberOfRecords>2599</numberOfRecords><records><record><recordSchema>MARC21-xml</recordSchema><recordPacking>xml</recordPacking><recordData><record xmlns="http://www.loc.gov/MARC21/slim" type="Bibliographic">\n    <leader>00000nam a2200000uc 4500</leader>\n    <controlfield tag="001">1418803235</controlfield>\n    <controlfield tag="003">DE-101</controlfield>\n    <controlfield tag="005">20261007225629.0</controlfield>\n    <controlfield tag="007">cr||||||||||||</controlfield>\n    <controlfield tag="008">260909s2026    gw |||||o|||| 00||||ger  </controlfield>\n    <datafield tag="015" ind1=" " ind2=" ">\n      <subfield code="a">26,O10</subfield>\n      <subfield code="2">dnb</subfield>\n    </datafield>\n    <datafield tag="016" ind1="7" ind2=" ">\n      <subfield code="2">DE-101</subfield>\n      <subfield code="a">1418803235</subfield>\n    </datafield>\n   

In [8]:
testresponse = soup(testresult.content, features="xml")
    
records = testresponse.find_all('record', {'type':'Bibliographic'})
print(len(records))
results = testresponse.find('numberOfRecords')  
     
numberofrecords = results.text
numberofrecords = int(numberofrecords)
print("Gefundene Treffer:", numberofrecords)

100
Gefundene Treffer: 2599


In [9]:
print(testresponse.prettify())

<?xml version="1.0" encoding="utf-8"?>
<searchRetrieveResponse xmlns="http://www.loc.gov/zing/srw/">
 <version>
  1.1
 </version>
 <numberOfRecords>
  2599
 </numberOfRecords>
 <records>
  <record>
   <recordSchema>
    MARC21-xml
   </recordSchema>
   <recordPacking>
    xml
   </recordPacking>
   <recordData>
    <record type="Bibliographic" xmlns="http://www.loc.gov/MARC21/slim">
     <leader>
      00000nam a2200000uc 4500
     </leader>
     <controlfield tag="001">
      1418803235
     </controlfield>
     <controlfield tag="003">
      DE-101
     </controlfield>
     <controlfield tag="005">
      20261007225629.0
     </controlfield>
     <controlfield tag="007">
      cr||||||||||||
     </controlfield>
     <controlfield tag="008">
      260909s2026    gw |||||o|||| 00||||ger
     </controlfield>
     <datafield ind1=" " ind2=" " tag="015">
      <subfield code="a">
       26,O10
      </subfield>
      <subfield code="2">
       dnb
      </subfield>
     </datafield>
    

In [14]:
# Funktion mit Schleife: 

def dnb_sru(query, mdformat):
    
    base_url = "https://services.dnb.de/sru/dnb"
    params = {'recordSchema' : mdformat,
          'operation': 'searchRetrieve',
          'version': '1.1',
          'maximumRecords': '100',
          'query': query
         }
    
    i = 0 
    records = []

    while i % 100 == 0: 
        params.update({'startRecord': i})       
        r = requests.get(base_url, params=params)
        xml = soup(r.content, features="xml")
        result = xml.find_all('record', {'type':'Bibliographic'})
        records+=result
        num_results = len(result)
        if num_results != 100:
            break
        else: 
            i += 100  
        
    return records

In [16]:
result = dnb_sru("sw=Kreuzzüge", "MARC21-xml")
print("Gesammelte Treffer: ", len(result))

Gesammelte Treffer:  706


### Speichern:

In [18]:
# mit Hilfe einer Schleife wird jeder Datensatz nacheinander in die Datei "datadump.xml" geschrieben. An den Anfang und das Ende der Datei werden zudem die beiden Texte 
# hinter "f.write" geschrieben, die eine Art Klammer um die XML-Datensätze bilden und so dafür sorgen, einige Basisinformationen mitzugeben und das XML valide zu halten. 

with open ("datadump.xml", "w", encoding="utf-8") as f:
    f.write(f'''<?xml version="1.0" encoding="utf-8"?><searchRetrieveResponse xmlns="http://www.loc.gov/zing/srw/"> 
                <version>1.1</version><numberOfRecords>{str(len(records))}</numberOfRecords>
                <records><record><recordSchema>MARC21-xml</recordSchema><recordPacking>xml</recordPacking><recordData>''')
    for record in result:
        f.write(str(record))
    f.write(f'''</recordData></record></records><echoedSearchRetrieveRequest><version>1.1</version>
                <xQuery xmlns:xsi="http://www.w3.org/2001/XMLSchema-instance" xsi:nil="true"/>
                <recordSchema>MARC21-xml</recordSchema></echoedSearchRetrieveRequest>
                </searchRetrieveResponse>''')